# 🎯 Module 11: Time Series

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 11 — Time Series  
> **Date:** 22 January 2026  
> **Previous:** 10 — Feature Selection  
> **Next:** 12 — Recommendation Systems

---

**Time series** data is collected or recorded **over time** — daily sales, stock prices, temperature, traffic, etc.

Unlike normal ML datasets, **the order of observations matters**.

In this notebook we'll cover:

- Components: **trend, seasonality, noise**
- Why random splits cause **leakage**
- **ARIMA (p, d, q)**
- **SARIMA (p,d,q)(P,D,Q,s)**
- **Prophet**
- **LSTM basics** (a peek)
- **TimeSeriesSplit**
- Forecast metrics: **MAE, RMSE, MAPE**

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Identify **trend**, **seasonality**, and **noise** in a time series
- Explain why random train/test splits cause **data leakage**
- Understand **stationarity** and **differencing**
- Fit an **ARIMA(p,d,q)** model
- Extend to **SARIMA(p,d,q)(P,D,Q,s)** for seasonal data
- Use **Prophet** for a quick baseline forecast
- Understand where **LSTM** fits in forecasting
- Use **TimeSeriesSplit** for honest validation
- Evaluate forecasts with **MAE, RMSE, MAPE**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Build a synthetic time series: trend + yearly seasonality + noise
np.random.seed(42)
n_days = 3 * 365
dates = pd.date_range(start='2023-01-01', periods=n_days, freq='D')

trend      = np.linspace(100, 200, n_days)                    # slow upward trend
seasonality = 20 * np.sin(2 * np.pi * np.arange(n_days) / 365)  # yearly cycle
noise      = np.random.normal(0, 8, n_days)

values = trend + seasonality + noise

df = pd.DataFrame({'date': dates, 'value': values})
print(df.head())
print(f'\nShape: {df.shape}')

In [ ]:
# Plot the raw series
plt.figure(figsize=(11, 4))
plt.plot(df['date'], df['value'], linewidth=0.8)
plt.title('Synthetic time series (trend + yearly seasonality + noise)')
plt.xlabel('Date'); plt.ylabel('Value')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🧠 1. Components of a Time Series

A time series often has three components:

### Trend
Long-term upward or downward movement.

```text
100 → 110 → 125 → 140 → 155
```

### Seasonality
A pattern that repeats at a **known interval**.

```text
Higher sales every December
Higher traffic every weekend
```

### Noise
Random variation that cannot be explained.

```text
Actual = Pattern + Random Variation
```

In [ ]:
# Decompose the series into trend, seasonal, and residual components
from statsmodels.tsa.seasonal import seasonal_decompose

series = df.set_index('date')['value']
decomposition = seasonal_decompose(series, model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
decomposition.observed.plot(ax=axes[0]); axes[0].set_ylabel('Observed')
decomposition.trend.plot(ax=axes[1]);    axes[1].set_ylabel('Trend')
decomposition.seasonal.plot(ax=axes[2]); axes[2].set_ylabel('Seasonal')
decomposition.resid.plot(ax=axes[3]);    axes[3].set_ylabel('Residual')
for ax in axes: ax.grid(alpha=0.3)
plt.suptitle('Time series decomposition')
plt.tight_layout(); plt.show()

---
## ⚠️ 2. Time Series vs Normal ML

For normal ML, random `train_test_split` is fine.

For time series, **mixing past and future** causes **data leakage**.

### ❌ Incorrect

```text
2024 ──┐
2025 ──┼── Random Split
2026 ──┘
```

### ✅ Correct

```text
Training              Test
2024 ───────────────→ 2025
```

> The model must learn from the **past** and predict the **future**.

In [ ]:
# Chronological train/test split (80/20)
split_idx = int(0.8 * len(df))
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f'Train: {train["date"].min().date()} → {train["date"].max().date()}  ({len(train)} rows)')
print(f'Test : {test["date"].min().date()} → {test["date"].max().date()}   ({len(test)} rows)')

plt.figure(figsize=(11, 4))
plt.plot(train['date'], train['value'], label='train')
plt.plot(test['date'],  test['value'],  label='test', color='orange')
plt.title('Chronological train / test split')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📐 3. ARIMA (p, d, q)

**ARIMA = AutoRegressive Integrated Moving Average**

$$ARIMA(p, d, q)$$

| Term | Meaning |
| --- | --- |
| **AR (p)** | Uses previous values (autoregression) |
| **I (d)** | Differencing to make the series stationary |
| **MA (q)** | Uses previous forecast errors (moving average) |

### Intuition

- **AR** — today's value depends on yesterday's value
- **I** — subtract consecutive values to stabilize the mean
- **MA** — use past errors to smooth the forecast

### Stationarity

ARIMA assumes the series is (approximately) **stationary** — its mean, variance, and autocorrelation don't change over time. **Differencing** (`d ≥ 1`) is the usual trick.

In [ ]:
# Check stationarity with the Augmented Dickey-Fuller test
from statsmodels.tsa.stattools import adfuller

def adf_report(series, name='series'):
    result = adfuller(series.dropna())
    print(f'{name}:')
    print(f'  ADF statistic: {result[0]:.4f}')
    print(f'  p-value      : {result[1]:.4f}')
    if result[1] < 0.05:
        print('  → reject H0 → likely stationary')
    else:
        print('  → fail to reject H0 → likely non-stationary')
    print()

adf_report(train['value'], 'Raw series')
adf_report(train['value'].diff(), 'First difference')

In [ ]:
# Fit ARIMA and forecast
from statsmodels.tsa.arima.model import ARIMA

# p=2, d=1, q=2 as a reasonable default for this synthetic series
arima_model = ARIMA(train['value'], order=(2, 1, 2))
arima_fit = arima_model.fit()

print(arima_fit.summary().tables[0])

In [ ]:
# Forecast the test window
forecast = arima_fit.forecast(steps=len(test))
forecast.index = test['date'].values

mae  = mean_absolute_error(test['value'], forecast)
rmse = np.sqrt(mean_squared_error(test['value'], forecast))
mape = np.mean(np.abs((test['value'].values - forecast.values) / test['value'].values)) * 100

print(f'ARIMA(2,1,2) — MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:.2f}%')

plt.figure(figsize=(11, 4))
plt.plot(train['date'], train['value'], label='train')
plt.plot(test['date'],  test['value'],  label='test', color='orange')
plt.plot(test['date'],  forecast,       label='ARIMA forecast', color='green', linestyle='--')
plt.title('ARIMA forecast on the test window')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Fit SARIMA (period = 7 to keep training time reasonable)
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima = SARIMAX(
    train['value'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima.fit(disp=False)

sarima_forecast = sarima_fit.forecast(steps=len(test))
sarima_forecast.index = test['date'].values

mae_s  = mean_absolute_error(test['value'], sarima_forecast)
rmse_s = np.sqrt(mean_squared_error(test['value'], sarima_forecast))

print(f'SARIMA(1,1,1)(1,1,1,7) — MAE={mae_s:.3f}  RMSE={rmse_s:.3f}')

---
## 🔮 5. Prophet

**Prophet** is a forecasting framework designed to be **simple and robust** for real-world data.

It handles:
- Trend
- Seasonality
- Holidays / events
- Missing observations
- Trend changes

### Prophet expects two columns

```text
ds → date/time
y  → value
```

In [ ]:
# Prophet — install with: pip install prophet
try:
    from prophet import Prophet

    prophet_train = train.rename(columns={'date': 'ds', 'value': 'y'})

    m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    m.fit(prophet_train)

    future = m.make_future_dataframe(periods=len(test))
    prophet_forecast = m.predict(future)

    # Align with the test window
    preds = prophet_forecast.iloc[-len(test):]['yhat'].values
    mae_p  = mean_absolute_error(test['value'], preds)
    rmse_p = np.sqrt(mean_squared_error(test['value'], preds))

    print(f'Prophet — MAE={mae_p:.3f}  RMSE={rmse_p:.3f}')

    plt.figure(figsize=(11, 4))
    plt.plot(train['date'], train['value'], label='train')
    plt.plot(test['date'],  test['value'],  label='test', color='orange')
    plt.plot(test['date'],  preds,          label='Prophet forecast', color='green', linestyle='--')
    plt.title('Prophet forecast on the test window')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print('Prophet not installed. Run: pip install prophet')

---
## 🧬 6. LSTM for Time Series (Overview)

**LSTM (Long Short-Term Memory)** is a recurrent neural network that can learn complex sequential patterns.

```text
Past 7 days
    ↓
LSTM
    ↓
Next day's prediction
```

Example input/output:

```text
Input:  [100, 110, 120, 115, 130, 140, 150]
Output: 155
```

### ⚠️ Note

LSTM belongs primarily to **Deep Learning**. Detailed architecture, gates, and backpropagation belong in that curriculum.

Here, we just want to know **where it fits** in a forecasting workflow.

### A tiny LSTM sketch (PyTorch)

```python
import torch.nn as nn

class TinyLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # use last timestep
```

Input shape: `[batch, sequence, features]`  
For univariate: `features = 1`.

---
## 🔁 7. TimeSeriesSplit

Normal K-Fold shuffles data — for time series, that's **leakage**.

**TimeSeriesSplit** preserves order: the training set grows forward, the validation set is always **after** it.

```python
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
```

In [ ]:
# Visualize TimeSeriesSplit folds
tscv = TimeSeriesSplit(n_splits=5)
X_dummy = df[['value']].values

fig, ax = plt.subplots(figsize=(10, 4))
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_dummy)):
    ax.scatter(tr_idx, [i] * len(tr_idx), color='steelblue', s=2, label='train' if i == 0 else '')
    ax.scatter(val_idx, [i] * len(val_idx), color='orange',    s=2, label='val'   if i == 0 else '')
ax.set_xlabel('Sample index'); ax.set_ylabel('Fold')
ax.set_title('TimeSeriesSplit folds — train grows forward, val follows')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📊 8. Forecasting Metrics

### MAE
$$MAE = \frac{1}{n} \sum |y_i - \hat{y}_i|$$
Average absolute prediction error.

### RMSE
$$RMSE = \sqrt{\frac{1}{n} \sum (y_i - \hat{y}_i)^2}$$
**Penalizes larger errors more strongly.**

### MAPE
$$MAPE = \frac{100}{n} \sum \left| \frac{y_i - \hat{y}_i}{y_i} \right|$$
Error as a **percentage**.

> ⚠️ MAPE can behave poorly when actual values are **zero or very close to zero**.

In [ ]:
# Compare all three metrics on the ARIMA forecast
def forecast_metrics(y_true, y_pred, name='model'):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    print(f'{name:<22} MAE={mae:>7.3f}  RMSE={rmse:>7.3f}  MAPE={mape:>7.2f}%')

forecast_metrics(test['value'], forecast,        'ARIMA')
forecast_metrics(test['value'], sarima_forecast, 'SARIMA')

try:
    forecast_metrics(test['value'], preds, 'Prophet')
except NameError:
    pass

---
## 🧪 9. Basic Forecasting Workflow

```text
Time-Series Data
      ↓
Sort by Date
      ↓
Explore Trend & Seasonality
      ↓
Check Missing Values
      ↓
Train/Test Split by Time
      ↓
Choose Model (ARIMA / SARIMA / Prophet / LSTM)
      ↓
Train
      ↓
Forecast Future
      ↓
Evaluate
      ↓
Improve
```

Always ensure dates are **sorted** and converted to `datetime`.

In [ ]:
# Sort + convert dates — the first thing to do with any real time-series data
df_sorted = df.copy()
df_sorted['date'] = pd.to_datetime(df_sorted['date'])
df_sorted = df_sorted.sort_values('date').reset_index(drop=True)

print('First date:', df_sorted['date'].iloc[0].date())
print('Last date :', df_sorted['date'].iloc[-1].date())
print('Any missing values:', df_sorted.isna().any().any())

---
## 📝 10. Practice Checklist

- [ ] Load a daily sales dataset
- [ ] Convert date column to `datetime`
- [ ] Sort observations chronologically
- [ ] Plot the time series
- [ ] Identify trend and seasonality
- [ ] Create a chronological train/test split
- [ ] Implement an ARIMA model
- [ ] Experiment with SARIMA on seasonal data
- [ ] Build a basic Prophet forecast
- [ ] Compare forecasts using MAE and RMSE
- [ ] Use `TimeSeriesSplit`
- [ ] Explain why random splits cause leakage

---
## 🏋️ 11. Hands-On Exercises

### Exercise 1 — ARIMA Grid
Try `order ∈ {(0,1,0), (1,1,0), (0,1,1), (1,1,1), (2,1,2), (3,1,3)}`. Which minimizes MAE on the test window?

### Exercise 2 — Seasonal Period Sweep
For SARIMA, try `s ∈ {7, 30, 365}`. Which seasonal period best fits the data?

### Exercise 3 — Prophet Holidays
Add `m.add_country_holidays(country_name='US')` and see if forecasts improve.

### Exercise 4 — TimeSeriesSplit + ARIMA
Use `TimeSeriesSplit` to evaluate ARIMA across multiple forward-chaining splits. Report mean and std of MAE.

### Exercise 5 — MAPE Pitfall
Construct a test window that contains a zero value. What happens to MAPE? Why?

### Exercise 6 — LSTM Baseline
Sketch a small PyTorch LSTM using sliding windows of length 30. Train it and compare MAE to ARIMA.

### Exercise 7 — Real Dataset
Download a real time series (e.g., Air Passengers, store sales) and repeat the full workflow.

---
## 🎤 12. Interview Questions

1. **What is time-series data?**  
   Observations recorded in chronological order where the sequence matters.

2. **Time series vs normal ML?**  
   Time series requires time-aware splits and respects order; normal ML can shuffle data.

3. **What are trend, seasonality, and noise?**  
   Trend = long-term movement. Seasonality = repeating pattern. Noise = unexplained randomness.

4. **What is stationarity?**  
   A series whose statistical properties (mean, variance, autocorrelation) don't change over time.

5. **What does ARIMA `(p, d, q)` mean?**  
   p = autoregressive terms. d = differencing. q = moving-average terms.

6. **ARIMA vs SARIMA?**  
   SARIMA adds seasonal AR/MA/I components — used when the series has a repeating seasonal pattern.

7. **What is seasonality?**  
   A repeating pattern at a fixed interval (weekly, monthly, yearly).

8. **What is Prophet?**  
   A framework for simple, robust forecasting with trend/seasonality/holiday handling.

9. **Why shouldn't we randomly shuffle time-series data?**  
   Because it mixes past and future → leakage → unrealistically optimistic evaluation.

10. **What is `TimeSeriesSplit`?**  
    A cross-validation scheme that preserves chronological order.

11. **When might LSTM be useful for forecasting?**  
    When the series has complex, non-linear, and long-range temporal patterns.

12. **Which metrics can be used to evaluate forecasts?**  
    MAE, RMSE, MAPE (careful with MAPE when values are near zero).

---
## 🏁 13. Key Takeaways

- **Time series** = observations ordered by time
- Forecasting predicts **future values from historical data**
- **Trend** → long-term movement
- **Seasonality** → repeating patterns
- **ARIMA** → autoregressive + differencing + moving average
- **SARIMA** → ARIMA + seasonality
- **Prophet** → practical trend/seasonality/holiday forecasting
- **LSTM** → deep-learning model for complex sequential patterns
- **Never allow future information** into training
- Use **time-aware validation** like `TimeSeriesSplit`
- Evaluate with **MAE, RMSE, MAPE**

### 🧭 Golden Rule

> **In time series, order is everything. Split by time, validate by time, and never let the future leak into the past.**

---

### 🔗 What's Next?

**Module 12 — Recommendation Systems**

We'll cover **content-based**, **collaborative**, and **hybrid** recommenders, plus **matrix factorization** with SVD and ALS.